# Closed trade history — Bybit (last N days, M5)

Fetches **closed P&L** for `SYMBOLS`, saves `./data/<SYMBOL>/trades/closed_pnl_history.csv`, and draws **one chart per symbol**: **M5 close** over the last **5 days** (default), with entry / exit markers and dashed connectors.

**Params:** `BYBIT_TRADE_LOOKBACK_DAYS` (default `5`), `BYBIT_TRADE_CHART_INTERVAL` (default `5` = M5). Bybit closed-PnL requests use at most a **7-day** `startTime`–`endTime` range per call; longer lookbacks are stepped in slices. Platform max history (~2 years) is still capped via `BYBIT_CLOSED_PNL_PLATFORM_CAP_DAYS` if you raise the lookback.

Requires `.env` with API keys; use **`BYBIT_DEMO=true`** for Demo Trading (`api-demo.bybit.com`).

In [8]:
%pip install plotly nbformat --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# SECTION 1 — Imports, symbols, paths
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import time
import hashlib
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

try:
    from dotenv import load_dotenv
    _env = _ROOT / ".env"
    if _env.exists():
        load_dotenv(_env)
        print(f"Loaded .env from {_env}")
    else:
        print(f"No .env at {_env}")
except ImportError:
    print("python-dotenv not installed")

SYMBOLS = [
    "BTCUSDT", "XRPUSDT", "ETHUSDT", "BNBUSDT", "ADAUSDT", "SOLUSDT", "DOGEUSDT",
    "DOTUSDT", "AVAXUSDT", "BCHUSDT", "SHIB1000USDT", "LTCUSDT", "XLMUSDT",
]

CATEGORY = "linear"
TESTNET = os.getenv("BYBIT_TESTNET", "false").lower() == "true"
DEMO = os.getenv("BYBIT_DEMO", "true").lower() == "true"

LOOKBACK_DAYS = int(os.getenv("BYBIT_TRADE_LOOKBACK_DAYS", "5"))
CLOSED_PNL_CAP_DAYS = int(os.getenv("BYBIT_CLOSED_PNL_PLATFORM_CAP_DAYS", "719"))
EFFECTIVE_CLOSED_PNL_DAYS = min(LOOKBACK_DAYS, CLOSED_PNL_CAP_DAYS)

CHART_INTERVAL = os.getenv("BYBIT_TRADE_CHART_INTERVAL", "5")

_TEHRAN = "Asia/Tehran"
BASE_DIR = Path("./data")

_mode = "demo" if DEMO else ("testnet" if TESTNET else "mainnet")
if DEMO and TESTNET:
    raise ValueError("Use only one of BYBIT_DEMO or BYBIT_TESTNET")

print(f"Symbols: {SYMBOLS}")
print(f"Endpoint: {_mode}  |  M{CHART_INTERVAL} chart  |  lookback {LOOKBACK_DAYS}d")

In [10]:
# SECTION 2 — Bybit client
from pybit.unified_trading import HTTP

api_key = os.getenv("BYBIT_API_KEY", "")
api_secret = os.getenv("BYBIT_API_SECRET", "")

client = HTTP(
    testnet=TESTNET,
    demo=DEMO,
    api_key=api_key or None,
    api_secret=api_secret or None,
)

print(f"HTTP demo={DEMO} testnet={TESTNET}  |  API key set: {bool(api_key)}")

HTTP demo=True testnet=False  |  API key set: True


In [ ]:
# SECTION 3 — Fetch helpers

SEVEN_MS = 7 * 24 * 60 * 60 * 1000
DAY_MS = 24 * 60 * 60 * 1000

# Bybit stopOrderType values (not always present in closed-pnl response)
_SL_TYPES = {"StopLoss", "Stop", "OcoTriggerByStopLoss"}
_TP_TYPES = {"TakeProfit", "OcoTriggerByTp"}


def _dedupe_key(row: dict) -> str:
    raw = "|".join(
        str(row.get(k, ""))
        for k in ("orderId", "symbol", "updatedTime", "closedSize", "avgExitPrice", "closedPnl")
    )
    return hashlib.sha256(raw.encode()).hexdigest()


def fetch_closed_pnl_all_history(symbol: str) -> list[dict]:
    now_ms = int(time.time() * 1000)
    boundary = now_ms - EFFECTIVE_CLOSED_PNL_DAYS * DAY_MS
    out: list[dict] = []
    seen: set[str] = set()

    window_end = now_ms
    while window_end > boundary:
        window_start = max(window_end - SEVEN_MS, boundary)
        cursor = None
        while True:
            kwargs = dict(
                category=CATEGORY, symbol=symbol, limit=100,
                startTime=str(window_start), endTime=str(window_end),
            )
            if cursor:
                kwargs["cursor"] = cursor

            resp = client.get_closed_pnl(**kwargs)
            if resp.get("retCode") != 0:
                raise RuntimeError(f"Bybit {resp.get('retCode')}: {resp.get('retMsg')}")

            result = resp["result"]
            items = result.get("list") or []
            for row in items:
                k = _dedupe_key(row)
                if k not in seen:
                    seen.add(k)
                    out.append(row)

            cursor = result.get("nextPageCursor") or ""
            if not cursor or not items:
                break
            time.sleep(0.06)

        window_end = window_start - 1
        time.sleep(0.06)

    return out


def closed_pnl_rows_to_dataframe(rows: list[dict]) -> pd.DataFrame:
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    for col in ("createdTime", "updatedTime", "createdAt", "updatedAt"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    t_entry_col = "createdTime" if "createdTime" in df.columns else "createdAt"
    t_exit_col  = "updatedTime" if "updatedTime" in df.columns else "updatedAt"
    if t_entry_col not in df.columns or t_exit_col not in df.columns:
        raise ValueError(f"Unexpected columns: {list(df.columns)}")

    df["entry_time"] = pd.to_datetime(df[t_entry_col], unit="ms", utc=True)
    df["exit_time"]  = pd.to_datetime(df[t_exit_col],  unit="ms", utc=True)
    for col in ("avgEntryPrice", "avgExitPrice", "closedPnl", "closedSize"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # side in closed-pnl is the CLOSING side — infer position direction (opposite)
    if "side" in df.columns:
        df["position"] = df["side"].map({"Buy": "Short", "Sell": "Long"}).fillna(df["side"])

    df = df.sort_values("exit_time").reset_index(drop=True)
    return df


def fetch_klines_in_range(symbol: str, interval: str, start_ms: int, end_ms: int) -> pd.DataFrame:
    all_rows: list[list] = []
    cur_end = int(end_ms)
    MAX_RETRIES = 5

    while cur_end >= start_ms:
        kwargs = dict(category=CATEGORY, symbol=symbol, interval=interval, limit=1000, end=cur_end)
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                resp = client.get_kline(**kwargs)
                if resp.get("retCode") != 0:
                    raise RuntimeError(resp.get("retMsg"))
                rows = resp["result"].get("list", [])
                break
            except Exception:
                if attempt == MAX_RETRIES:
                    raise
                time.sleep(2 ** attempt)

        if not rows:
            break
        all_rows.extend(rows)
        oldest = int(rows[-1][0])
        if oldest <= start_ms or len(rows) < 1000:
            break
        cur_end = oldest - 1
        time.sleep(0.05)

    if not all_rows:
        return pd.DataFrame(columns=["open", "high", "low", "close", "volume"])

    df = pd.DataFrame(
        list(reversed(all_rows)),
        columns=["time", "open", "high", "low", "close", "volume", "turnover"],
    )
    df["time"] = pd.to_datetime(df["time"].astype("int64"), unit="ms", utc=True)
    df = df.set_index("time").sort_index()
    df = df[~df.index.duplicated(keep="last")]
    for col in ("open", "high", "low", "close", "volume"):
        df[col] = df[col].astype(float)

    start_dt = pd.to_datetime(start_ms, unit="ms", utc=True)
    end_dt   = pd.to_datetime(end_ms,   unit="ms", utc=True)
    return df.loc[(df.index >= start_dt) & (df.index <= end_dt),
                  ["open", "high", "low", "close", "volume"]]


def _classify_exit(row: pd.Series) -> str:
    sot = str(row.get("stopOrderType", "") or "").strip()
    if sot in _SL_TYPES:
        return "SL"
    if sot in _TP_TYPES:
        return "TP"
    try:
        return "SL" if float(row.get("closedPnl", 0)) < 0 else "TP"
    except (TypeError, ValueError):
        return "manual"


def plot_symbol_trades(symbol: str, ohlcv: pd.DataFrame, trades: pd.DataFrame) -> None:
    fig = go.Figure()

    # Candlestick — every bar is hoverable with O/H/L/C
    if not ohlcv.empty:
        fig.add_trace(go.Candlestick(
            x=ohlcv.index,
            open=ohlcv["open"], high=ohlcv["high"],
            low=ohlcv["low"],  close=ohlcv["close"],
            name="Price",
            increasing_line_color="#26a69a",
            decreasing_line_color="#ef5350",
            hovertext=[
                f"O: {o:.4f}  H: {h:.4f}  L: {l:.4f}  C: {c:.4f}"
                for o, h, l, c in zip(ohlcv["open"], ohlcv["high"], ohlcv["low"], ohlcv["close"])
            ],
            hoverinfo="x+text",
        ))

    if not trades.empty:
        trades = trades.copy()
        trades["_exit_type"] = trades.apply(_classify_exit, axis=1)
        et = trades["entry_time"].dt.tz_convert(_TEHRAN)
        xt = trades["exit_time"].dt.tz_convert(_TEHRAN)

        # Dashed connectors
        for i in range(len(trades)):
            fig.add_trace(go.Scatter(
                x=[et.iloc[i], xt.iloc[i]],
                y=[float(trades["avgEntryPrice"].iloc[i]), float(trades["avgExitPrice"].iloc[i])],
                mode="lines",
                line=dict(color="rgba(150,150,150,0.5)", width=1, dash="dash"),
                showlegend=False, hoverinfo="skip",
            ))

        # Entry markers — triangle-up arrow
        pos_col  = trades["position"].tolist()  if "position"  in trades.columns else [""] * len(trades)
        ord_col  = trades["orderType"].tolist() if "orderType" in trades.columns else [""] * len(trades)
        fig.add_trace(go.Scatter(
            x=et, y=trades["avgEntryPrice"],
            mode="markers", name="Entry",
            marker=dict(symbol="triangle-up", color="royalblue", size=13),
            customdata=list(zip(trades["avgEntryPrice"].round(4), pos_col, ord_col)),
            hovertemplate=(
                "<b>Entry</b><br>"
                "Time: %{x}<br>"
                "Price: %{customdata[0]}<br>"
                "Position: %{customdata[1]}<br>"
                "OrderType: %{customdata[2]}"
                "<extra></extra>"
            ),
        ))

        # Exit — SL  (red filled X)
        sl_mask = trades["_exit_type"] == "SL"
        sl_t = trades[sl_mask]
        if len(sl_t):
            fig.add_trace(go.Scatter(
                x=xt[sl_mask], y=sl_t["avgExitPrice"],
                mode="markers", name="Exit SL",
                marker=dict(symbol="x", color="firebrick", size=13,
                            line=dict(width=2.5, color="firebrick")),
                customdata=list(zip(sl_t["avgExitPrice"].round(4), sl_t["closedPnl"].round(4))),
                hovertemplate=(
                    "<b>Exit — Stop Loss</b><br>"
                    "Time: %{x}<br>"
                    "Price: %{customdata[0]}<br>"
                    "PnL: %{customdata[1]}"
                    "<extra></extra>"
                ),
            ))

        # Exit — TP  (green circle)
        tp_mask = trades["_exit_type"] == "TP"
        tp_t = trades[tp_mask]
        if len(tp_t):
            fig.add_trace(go.Scatter(
                x=xt[tp_mask], y=tp_t["avgExitPrice"],
                mode="markers", name="Exit TP",
                marker=dict(symbol="circle", color="seagreen", size=12),
                customdata=list(zip(tp_t["avgExitPrice"].round(4), tp_t["closedPnl"].round(4))),
                hovertemplate=(
                    "<b>Exit — Take Profit</b><br>"
                    "Time: %{x}<br>"
                    "Price: %{customdata[0]}<br>"
                    "PnL: %{customdata[1]}"
                    "<extra></extra>"
                ),
            ))

    fig.update_layout(
        title=f"{symbol}  |  last {LOOKBACK_DAYS}d  M{CHART_INTERVAL}  |  closed trades: {len(trades)}",
        xaxis_title="Time (Tehran)", yaxis_title="Price",
        hovermode="closest", height=520,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        xaxis=dict(
            rangeslider=dict(visible=True, thickness=0.05),
            rangeselector=dict(buttons=[
                dict(count=6,  label="6h",  step="hour", stepmode="backward"),
                dict(count=1,  label="1D",  step="day",  stepmode="backward"),
                dict(count=3,  label="3D",  step="day",  stepmode="backward"),
                dict(step="all", label="All"),
            ]),
            type="date",
        ),
        margin=dict(t=80, b=20),
    )
    display(fig)   # display() works in VS Code Jupyter; fig.show() does not


print("Helpers ready.")

In [12]:
# SECTION 4 — Download closed P&L for every symbol and save CSV
trade_frames: dict[str, pd.DataFrame] = {}

for symbol in SYMBOLS:
    print(f"Fetching closed P&L: {symbol} ...")
    try:
        rows = fetch_closed_pnl_all_history(symbol)
        df = closed_pnl_rows_to_dataframe(rows)
        trade_frames[symbol] = df
        out_dir = BASE_DIR / symbol / "trades"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_csv = out_dir / "closed_pnl_history.csv"
        df.to_csv(out_csv, index=False)
        print(f"  -> {len(df)} rows  saved {out_csv}")
    except Exception as exc:
        print(f"  FAILED: {exc}")
        trade_frames[symbol] = pd.DataFrame()

print("Done fetching.")

Fetching closed P&L: BTCUSDT ...
  -> 8 rows  saved data\BTCUSDT\trades\closed_pnl_history.csv
Fetching closed P&L: XRPUSDT ...
  -> 2 rows  saved data\XRPUSDT\trades\closed_pnl_history.csv
Fetching closed P&L: ETHUSDT ...
  -> 1 rows  saved data\ETHUSDT\trades\closed_pnl_history.csv
Fetching closed P&L: BNBUSDT ...
  -> 6 rows  saved data\BNBUSDT\trades\closed_pnl_history.csv
Fetching closed P&L: ADAUSDT ...
  -> 3 rows  saved data\ADAUSDT\trades\closed_pnl_history.csv
Fetching closed P&L: SOLUSDT ...
  -> 14 rows  saved data\SOLUSDT\trades\closed_pnl_history.csv
Fetching closed P&L: DOGEUSDT ...
  -> 5 rows  saved data\DOGEUSDT\trades\closed_pnl_history.csv
Fetching closed P&L: DOTUSDT ...
  -> 3 rows  saved data\DOTUSDT\trades\closed_pnl_history.csv
Fetching closed P&L: AVAXUSDT ...
  -> 2 rows  saved data\AVAXUSDT\trades\closed_pnl_history.csv
Fetching closed P&L: BCHUSDT ...
  -> 1 rows  saved data\BCHUSDT\trades\closed_pnl_history.csv
Fetching closed P&L: SHIB1000USDT ...
  -> 1 

In [13]:
# SECTION 4b — Inspect raw trade fields (run once to verify timestamps & columns)
_debug_sym = "BTCUSDT"
_df = trade_frames.get(_debug_sym, pd.DataFrame())
if not _df.empty:
    show_cols = [c for c in [
        "entry_time", "exit_time", "avgEntryPrice", "avgExitPrice",
        "closedPnl", "side", "orderType", "stopOrderType", "orderId",
    ] if c in _df.columns]
    print(f"{_debug_sym}  columns: {list(_df.columns)}\n")
    with pd.option_context("display.max_columns", None, "display.width", 200):
        print(_df[show_cols].tail(5).to_string())
else:
    print(f"No trades for {_debug_sym}")

BTCUSDT  columns: ['symbol', 'orderType', 'leverage', 'updatedTime', 'side', 'orderId', 'closedPnl', 'openFee', 'closeFee', 'avgEntryPrice', 'qty', 'cumEntryValue', 'createdTime', 'orderPrice', 'closedSize', 'avgExitPrice', 'execType', 'fillCount', 'cumExitValue', 'entry_time', 'exit_time']

                        entry_time                        exit_time  avgEntryPrice  avgExitPrice  closedPnl side orderType                               orderId
3 2026-05-13 18:48:19.396000+00:00 2026-05-13 22:05:19.991000+00:00        79515.1       79246.7  13.943238  Buy    Market  66146af8-40e2-4f02-a4e2-003c1f3f3128
4 2026-05-13 23:19:45.376000+00:00 2026-05-14 00:15:01.226000+00:00        79311.0       79509.6 -25.635246  Buy    Market  eec2a252-cbe9-48ea-8fb6-ade0b0c39344
5 2026-05-14 01:16:17.325000+00:00 2026-05-14 02:50:30.786000+00:00        79500.8       79334.6  12.799897  Buy    Market  56c7252f-ec0c-4610-8278-6155fb9560c9
6 2026-05-14 03:08:23.943000+00:00 2026-05-14 05:33:11.046000+0

In [14]:
# SECTION 5 — Last LOOKBACK_DAYS on M5: price + trades (one figure per symbol)
_now_ms = int(time.time() * 1000)
chart_t0 = _now_ms - LOOKBACK_DAYS * DAY_MS
chart_t1 = _now_ms
_cutoff = pd.to_datetime(chart_t0, unit="ms", utc=True)

for symbol in SYMBOLS:
    trades = trade_frames.get(symbol, pd.DataFrame())
    if not trades.empty:
        trades = trades.loc[trades["exit_time"] >= _cutoff].copy()

    print(
        f"OHLCV {symbol}  {pd.to_datetime(chart_t0, unit='ms', utc=True)} .. "
        f"{pd.to_datetime(chart_t1, unit='ms', utc=True)} UTC"
    )
    ohlcv = fetch_klines_in_range(symbol, CHART_INTERVAL, chart_t0, chart_t1)
    if not ohlcv.empty:
        ohlcv.index = ohlcv.index.tz_convert(_TEHRAN)

    plot_symbol_trades(symbol, ohlcv, trades)

print("All charts rendered.")

OHLCV BTCUSDT  2026-05-09 13:01:19.040000+00:00 .. 2026-05-14 13:01:19.040000+00:00 UTC


OHLCV XRPUSDT  2026-05-09 13:01:19.040000+00:00 .. 2026-05-14 13:01:19.040000+00:00 UTC


OHLCV ETHUSDT  2026-05-09 13:01:19.040000+00:00 .. 2026-05-14 13:01:19.040000+00:00 UTC


OHLCV BNBUSDT  2026-05-09 13:01:19.040000+00:00 .. 2026-05-14 13:01:19.040000+00:00 UTC


OHLCV ADAUSDT  2026-05-09 13:01:19.040000+00:00 .. 2026-05-14 13:01:19.040000+00:00 UTC


...

OHLCV BCHUSDT  2026-05-09 13:01:19.040000+00:00 .. 2026-05-14 13:01:19.040000+00:00 UTC


OHLCV SHIB1000USDT  2026-05-09 13:01:19.040000+00:00 .. 2026-05-14 13:01:19.040000+00:00 UTC


OHLCV LTCUSDT  2026-05-09 13:01:19.040000+00:00 .. 2026-05-14 13:01:19.040000+00:00 UTC


OHLCV XLMUSDT  2026-05-09 13:01:19.040000+00:00 .. 2026-05-14 13:01:19.040000+00:00 UTC


All charts rendered.
